# 03 - ML experiments

Reads the consolidated dataset only; never runs the simulator. Features exclude generation metadata (see `district_cooling_network.ml.features`), splits are by complete scenario (`pair_id`) or by `seed`. Each run lives in `results/ml/<name>/<config-hash>/` and is skipped when already complete.

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["SWIM_WORKSPACE"] = "/content/drive/MyDrive/DCN_Thesis"
    %pip install -q -e "/content/drive/MyDrive/DCN_Thesis/district-cooling-network"

import pandas as pd

from district_cooling_network.ml.features import add_labels, feature_matrix
from district_cooling_network.ml.splits import split_by_group
from district_cooling_network.ml.tracking import start_run
from district_cooling_network.paths import get_workspace

ws = get_workspace()
DATASET = "pilot_v1"
data = add_labels(pd.read_parquet(ws.dataset_dir(DATASET) / "consolidated" / "dataset.parquet"))
splits = split_by_group(data, "pair_id", rng_seed=0)
{k: (len(v), v.pair_id.nunique()) for k, v in splits.items()}

In [ ]:
config = {"dataset": DATASET, "model": "mlp", "split": "pair_id", "rng_seed": 0}
run = start_run("baseline", config)
if run.is_complete:
    print("already done:", run.path)
else:
    X_train, y_train = feature_matrix(splits["train"]), splits["train"]["fault_class"]
    # ... fit and evaluate your model here ...
    # run.write_metrics({...}); run.finish()